# Airbnb Price Prediction

## Objectif
PrÃ©dire le **prix de nuit** de logements Airbnb Ã  partir de leurs caractÃ©ristiques.

## DonnÃ©es
- **Train** : 22 235 annonces avec le prix connu
- **Test** : 51 881 annonces dont on doit prÃ©dire le prix
- Variables disponibles : type de logement, ville, quartier, Ã©quipements, infos hÃ´te, avis...

## Ã‰tapes du projet
1. **Explorer les donnÃ©es** â€” comprendre ce qui influence le prix
2. **PrÃ©parer les features** â€” transformer les colonnes brutes en nombres utilisables
3. **Encoder les catÃ©gories** â€” convertir les textes en valeurs numÃ©riques
4. **EntraÃ®ner un modÃ¨le** â€” LightGBM avec validation croisÃ©e
5. **Analyser les rÃ©sultats** â€” vÃ©rifier que le modÃ¨le est cohÃ©rent

> La variable cible est `log_price` (le logarithme du prix).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    USE_LGBM = True
    print(f"LightGBM {lgb.__version__} disponible")
except ImportError:
    USE_LGBM = False
    print("LightGBM non disponible â€” fallback GradientBoostingRegressor")

plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)
SEED = 42
np.random.seed(SEED)

## 1. Chargement des donnÃ©es

In [ ]:
train = pd.read_csv('airbnb_train.csv')
test  = pd.read_csv('airbnb_test.csv')

# La 1Ã¨re colonne du test est sans nom (c'est l'id)
test.rename(columns={test.columns[0]: 'id'}, inplace=True)

print(f"Train : {train.shape[0]:,} lignes x {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes x {test.shape[1]} colonnes")
train.head(3)

## 2. Exploration des donnÃ©es (EDA)

In [ ]:
# Valeurs manquantes
def missing_report(df, name):
    m = (df.isnull() | (df == '')).sum()
    m = m[m > 0].sort_values(ascending=False)
    pct = (m / len(df) * 100).round(1)
    print(f"\n--- {name} ---")
    print(pd.DataFrame({'manquants': m, '%': pct}).to_string())

missing_report(train, 'Train')

### Observations : valeurs manquantes

Certaines colonnes ont beaucoup de valeurs manquantes :
- `host_response_rate` : environ 40% des hÃ´tes ne renseignent pas ce champ
- `review_scores_rating`, `first_review`, `last_review` : vides pour les nouvelles annonces sans avis

**Ce qu'on va faire** : crÃ©er une colonne supplÃ©mentaire pour indiquer si la date est absente (c'est une info utile), et remplacer les valeurs numÃ©riques manquantes par la mÃ©diane.

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['log_price'], bins=60, color='steelblue', edgecolor='white', lw=0.5)
axes[0].axvline(train['log_price'].mean(), color='red', ls='--',
                label=f"Moyenne : {train['log_price'].mean():.2f}")
axes[0].set_title('Distribution de log_price'); axes[0].set_xlabel('log_price')
axes[0].legend()

axes[1].hist(np.exp(train['log_price']), bins=100, color='coral', edgecolor='white', lw=0.5)
axes[1].set_xlim(0, 800); axes[1].set_title('Distribution du prix ($)')
axes[1].set_xlabel('Prix ($)')

plt.suptitle('Variable cible', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()
print(f"min={train['log_price'].min():.2f}  max={train['log_price'].max():.2f}  "
      f"mean={train['log_price'].mean():.2f}  std={train['log_price'].std():.2f}")

In [ ]:
# Prix moyen par catÃ©gorie
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flatten(), ['room_type','property_type','city','cancellation_policy']):
    means = train.groupby(col)['log_price'].mean().sort_values(ascending=False)
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(means)))
    bars = ax.bar(range(len(means)), means.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'log_price moyen par {col}', fontsize=11)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

### Observations : prix par catÃ©gorie

- Louer un logement entier coÃ»te environ 2x plus cher qu'une chambre privÃ©e
- Bateaux, chÃ¢teaux et villas ont les prix les plus Ã©levÃ©s
- San Francisco et New York sont les villes les plus chÃ¨res
- Les hÃ´tes avec des rÃ¨gles d'annulation strictes proposent souvent des logements haut de gamme

Ces variables ont donc un vrai impact sur le prix et seront utiles au modÃ¨le.

In [ ]:
# CorrÃ©lations numÃ©riques
num_cols = ['accommodates','bathrooms','bedrooms','beds','number_of_reviews','review_scores_rating']
tmp = train[num_cols + ['log_price']].copy()
for c in num_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
corr = tmp.corr()['log_price'].drop('log_price').sort_values(key=abs, ascending=False)
print("CorrÃ©lations avec log_price:")
print(corr.round(3))

### Observations : corrÃ©lations numÃ©riques

- Plus le logement est grand (voyageurs, chambres, salles de bain), plus il est cher
- Les logements avec beaucoup d'avis sont souvent moins chers (les logements abordables tournent davantage)
- La note des avis a peu d'effet direct sur le prix

La taille du logement est donc le principal facteur numÃ©rique.

## 2.2 Visualisations AvancÃ©es

Pour approfondir la comprÃ©hension des donnÃ©es :
- **Carte gÃ©ographique** des annonces colorÃ©e par prix (rouge = cher, vert = pas cher)
- **Heatmap de corrÃ©lation** entre toutes les variables numÃ©riques
- **Boxplot** comparant la distribution des prix par ville
- **Impact des amenities** : quels Ã©quipements font monter ou baisser le prix ?
- **Violin plot** par type de logement + courbe prix / capacitÃ© d'accueil

In [ ]:
# â”€â”€ Carte gÃ©ographique des prix (rouge = cher, vert = pas cher) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
prices = np.exp(train['log_price'])
p5, p95 = np.percentile(prices, 5), np.percentile(prices, 95)

fig, ax = plt.subplots(figsize=(15, 9))
sc = ax.scatter(
    train['longitude'], train['latitude'],
    c=prices.clip(p5, p95),
    cmap='RdYlGn_r',
    s=6, alpha=0.45,
    vmin=p5, vmax=p95
)
cbar = plt.colorbar(sc, ax=ax, label='Prix par nuit ($)', fraction=0.03)
cbar.ax.tick_params(labelsize=9)

for city, grp in train.groupby('city'):
    cx, cy = grp['longitude'].median(), grp['latitude'].median()
    ax.annotate(city, (cx, cy), fontsize=10, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#333333', alpha=0.8))

ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Carte des annonces Airbnb â€” rouge = cher, vert = pas cher', fontsize=14, pad=12)
ax.set_facecolor('#e8e8e8')
fig.patch.set_facecolor('#f5f5f5')
plt.tight_layout()
plt.show()
print(f"Prix mÃ©dian : \${np.median(prices):.0f}/nuit  |  Prix max : \${prices.max():.0f}/nuit")

In [ ]:
# â”€â”€ Heatmap de corrÃ©lation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
num_cols_viz = ['log_price', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
                'number_of_reviews', 'review_scores_rating']
labels_fr = {'log_price': 'Prix (log)', 'accommodates': 'Voyageurs',
             'bathrooms': 'Salles de bain', 'bedrooms': 'Chambres', 'beds': 'Lits',
             'number_of_reviews': 'Nb avis', 'review_scores_rating': 'Note'}

tmp_corr = train[num_cols_viz].copy()
for c in num_cols_viz:
    tmp_corr[c] = pd.to_numeric(tmp_corr[c], errors='coerce')
tmp_corr.rename(columns=labels_fr, inplace=True)

corr = tmp_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.8, linecolor='white',
            annot_kws={'size': 10}, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de corrÃ©lation â€” variables numÃ©riques', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ Boxplot des prix par ville â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
city_order = (train.groupby('city')['log_price']
              .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(14, 6))
palette_cities = sns.color_palette('RdYlGn_r', n_colors=len(city_order))
sns.boxplot(data=train, x='city', y='log_price', order=city_order,
            palette=palette_cities, linewidth=1.2, fliersize=2, ax=ax)
ax.set_xticklabels(city_order, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('log_price (= ln du prix en $)', fontsize=11)
ax.set_xlabel('')
ax.set_title('Distribution des prix par ville â€” du plus cher au moins cher', fontsize=13)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ Amenities qui font monter / baisser le prix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _parse_am(s):
    if pd.isna(s): return []
    return [i.strip().strip('"') for i in str(s).strip('{}').split(',') if i.strip().strip('"')]

am_prices_dict = {}
for _, row in train.iterrows():
    for am in _parse_am(row['amenities']):
        if am not in am_prices_dict:
            am_prices_dict[am] = []
        am_prices_dict[am].append(row['log_price'])

am_df = pd.DataFrame([
    {'amenity': am, 'avg_log_price': np.mean(v), 'count': len(v)}
    for am, v in am_prices_dict.items() if len(v) >= 200
]).sort_values('avg_log_price', ascending=False)

global_mean_price = train['log_price'].mean()
top_am  = am_df.head(15).copy()
bot_am  = am_df.tail(10).copy()
plot_df = pd.concat([top_am, bot_am]).reset_index(drop=True)
plot_df['delta'] = plot_df['avg_log_price'] - global_mean_price
plot_df['color'] = plot_df['delta'].apply(lambda x: '#e74c3c' if x > 0 else '#2ecc71')

fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(range(len(plot_df)), plot_df['delta'].values,
        color=plot_df['color'].values, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['amenity'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Ã‰cart au prix moyen (en log_price)', fontsize=11)
ax.set_title('Amenities qui font monter (rouge) ou baisser (vert) le prix', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ Violin + courbe prix / voyageurs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

room_order = (train.groupby('room_type')['log_price']
              .median().sort_values(ascending=False).index.tolist())
sns.violinplot(data=train, x='room_type', y='log_price', order=room_order,
               palette='coolwarm', inner='quartile', ax=axes[0])
axes[0].set_title('Distribution fine par type de logement', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('log_price')
axes[0].tick_params(axis='x', rotation=15)

acc_tmp = train.copy()
acc_tmp['accommodates'] = pd.to_numeric(acc_tmp['accommodates'], errors='coerce')
acc_tmp = acc_tmp.dropna(subset=['accommodates'])
acc_tmp['acc_bin'] = acc_tmp['accommodates'].clip(1, 10).astype(int)
acc_means = acc_tmp.groupby('acc_bin')['log_price'].mean()

axes[1].plot(acc_means.index, acc_means.values, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[1].fill_between(acc_means.index, acc_means.values, alpha=0.15, color='steelblue')
axes[1].set_xlabel('Nombre de voyageurs', fontsize=11)
axes[1].set_ylabel('log_price moyen', fontsize=11)
axes[1].set_title('Prix moyen selon le nombre de voyageurs', fontsize=12)
axes[1].grid(alpha=0.4)
axes[1].set_xticks(acc_means.index)
plt.tight_layout()
plt.show()

### Ce qu'on retient de l'exploration

1. **La localisation** est clÃ© : deux logements dans la mÃªme ville mais dans des quartiers diffÃ©rents peuvent avoir des prix trÃ¨s diffÃ©rents
2. **La taille** du logement est fortement liÃ©e au prix
3. **Les Ã©quipements** peuvent faire monter le prix (doorman, salle de sport, jacuzzi...)
4. **Le type** de logement (entier vs chambre) est trÃ¨s discriminant

Ces observations nous guident pour choisir quelles features crÃ©er.

## 3. Feature Engineering

Le modÃ¨le ne peut lire que des nombres. On transforme donc les colonnes texte, dates et listes en valeurs numÃ©riques.

| Colonne brute | Transformation | Pourquoi |
|---------------|----------------|----------|
| `amenities` (liste d'Ã©quipements) | Une colonne 0/1 pour les **50 Ã©quipements les plus frÃ©quents** | Permet au modÃ¨le de savoir si le logement a la WiFi, un parking, etc. |
| `host_since`, `first_review`... (dates) | Nombre de jours avant 2018 + flag `_missing` | Le modÃ¨le ne lit pas les dates, mais un nombre oui |
| `host_has_profile_pic` (`t`/`f`) | 0 ou 1 | Conversion simple |
| `description`, `name` (texte) | Longueur + **TF-IDF + SVD** (cf. section 3.5) | Capture la sÃ©mantique au-delÃ  de la longueur |
| `host_response_rate` (`90%`) | 0.9 | Conversion en nombre dÃ©cimal |
| `zipcode`, `neighbourhood` | Nettoyage | Pour l'encodage qui suit |
| `accommodates`, `beds`, `bathrooms`, `bedrooms` | **Ratios d'interaction** (lits/personne, sdb/chambreâ€¦) | Capture la densitÃ© d'occupation |
| `latitude`, `longitude`, `city` | **Distance au centre** de la ville (mÃ©diane train) | Un logement central vaut souvent plus qu'en banlieue |

In [ ]:
def parse_amenities(s):
    if pd.isna(s) or s == '':
        return []
    items = []
    for item in str(s).strip('{}').split(','):
        item = item.strip().strip('"').strip()
        if item:
            items.append(item)
    return items

# Top 50 amenities (calculÃ© sur le train uniquement)
all_am = []
for a in train['amenities']:
    all_am.extend(parse_amenities(a))
am_counts = Counter(all_am)
TOP_AMENITIES = [a for a, _ in am_counts.most_common(50)]

print("Top 50 amenities :")
for i, (a, c) in enumerate(am_counts.most_common(50), 1):
    print(f"  {i:2d}. {a:<45} ({c:,})")

In [ ]:
def engineer_features(df, city_centers=None):
    # Applique toutes les transformations de features sur train ET test.
    # city_centers : dict ville -> (lat mÃ©diane, lon mÃ©diane) calculÃ© sur le train,
    # passÃ© aux deux appels pour Ã©viter le leak.
    df = df.copy()

    # â”€â”€ Amenities â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Colonne brute : {"TV","Wifi","Kitchen"} â€” on extrait chaque Ã©quipement
    # comme variable binaire (1 = prÃ©sent) pour les N plus frÃ©quents du train
    parsed_am = df['amenities'].apply(parse_amenities)
    df['amenities_count'] = parsed_am.apply(len)  # nb total d'Ã©quipements
    for am in TOP_AMENITIES:
        col = 'am_' + re.sub(r'[^a-z0-9]', '_', am.lower())[:30]
        df[col] = parsed_am.apply(lambda x: 1 if am in x else 0)

    # â”€â”€ Dates â†’ jours Ã©coulÃ©s â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # RÃ©fÃ©rence : 01/01/2018 (fin de la pÃ©riode des donnÃ©es)
    # On ajoute _missing car l'absence de date (pas d'avis) est une info utile
    ref = pd.Timestamp('2018-01-01')
    for dc in ['host_since', 'first_review', 'last_review']:
        parsed = pd.to_datetime(df[dc], errors='coerce')
        df[dc + '_days']    = (ref - parsed).dt.days
        df[dc + '_missing'] = parsed.isna().astype(int)

    # â”€â”€ BoolÃ©ens t/f â†’ 0/1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Le format CSV Airbnb stocke les boolÃ©ens comme 't' ou 'f'
    for c in ['host_has_profile_pic', 'host_identity_verified', 'instant_bookable']:
        df[c] = (df[c].astype(str).str.lower() == 't').astype(int)
    df['cleaning_fee'] = (df['cleaning_fee'].astype(str).str.lower() == 'true').astype(int)

    # â”€â”€ Taux de rÃ©ponse '90%' â†’ 0.9 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def parse_rate(x):
        try:
            return float(str(x).replace('%','').strip()) / 100.0
        except Exception:
            return np.nan
    df['host_response_rate'] = df['host_response_rate'].apply(
        lambda x: np.nan if pd.isna(x) or str(x).strip() == '' else parse_rate(x))

    # â”€â”€ Longueur du texte â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Proxy de qualitÃ© : une description longue et soignÃ©e = hÃ´te professionnel
    df['description_len']   = df['description'].fillna('').apply(len)
    df['description_words'] = df['description'].fillna('').apply(lambda x: len(x.split()))
    df['name_len']           = df['name'].fillna('').apply(len)

    # â”€â”€ Conversion numÃ©rique â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Certaines colonnes sont lues comme 'object' Ã  cause de valeurs mixtes
    for c in ['bathrooms','bedrooms','beds','review_scores_rating','number_of_reviews','accommodates']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # â”€â”€ Interactions de taille â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # DensitÃ© d'occupation : un studio pour 6 personnes vs villa pour 2 = trÃ¨s diffÃ©rent
    acc = df['accommodates'].replace(0, np.nan)
    bed = df['bedrooms'].replace(0, np.nan)
    df['beds_per_person']    = df['beds'] / acc
    df['bath_per_bedroom']   = df['bathrooms'] / bed
    df['bedroom_per_person'] = df['bedrooms'] / acc
    df['has_no_bathroom']    = (df['bathrooms'].fillna(0) == 0).astype(int)

    # â”€â”€ Distance au centre-ville (centre = mÃ©diane lat/lon par ville sur train) â”€
    if city_centers is not None:
        lat_map = df['city'].map({c: ll[0] for c, ll in city_centers.items()})
        lon_map = df['city'].map({c: ll[1] for c, ll in city_centers.items()})
        df['dist_to_center'] = np.sqrt(
            (df['latitude'] - lat_map) ** 2 + (df['longitude'] - lon_map) ** 2
        )
    else:
        df['dist_to_center'] = np.nan

    # â”€â”€ Indicateur : pas de reviews â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Les nouvelles annonces sans note ont un comportement de prix diffÃ©rent
    df['review_missing'] = df['review_scores_rating'].isna().astype(int)

    # â”€â”€ Nettoyage pour encodage â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Standardisation du zip (5 chiffres) et du quartier pour rÃ©duire le bruit
    df['zipcode_clean']       = df['zipcode'].fillna('Unknown').astype(str).str.strip().str[:5]
    df['neighbourhood_clean'] = df['neighbourhood'].fillna('Unknown').astype(str).str.strip()

    return df

# Centres mÃ©dians par ville, calculÃ©s sur le train uniquement (anti-leak)
_train_lat = pd.to_numeric(train['latitude'], errors='coerce')
_train_lon = pd.to_numeric(train['longitude'], errors='coerce')
city_centers = {
    c: (_train_lat[train['city'] == c].median(),
        _train_lon[train['city'] == c].median())
    for c in train['city'].dropna().unique()
}

train_fe = engineer_features(train, city_centers=city_centers)
test_fe  = engineer_features(test,  city_centers=city_centers)
print(f'Colonnes aprÃ¨s feature engineering : {train_fe.shape[1]}')
print(f'Centres villes (train): {len(city_centers)}')

## 3.5 Features texte avancÃ©es (TF-IDF + SVD)

Les longueurs de texte (`description_len`, `name_len`) capturent peu d'information. Pour aller plus loin, on extrait le **contenu sÃ©mantique** :

1. **TF-IDF** transforme chaque description en un vecteur oÃ¹ chaque dimension correspond Ã  un mot ou bi-gramme important (ex: "stunning view", "near subway"). Les mots frÃ©quents dans toutes les annonces sont automatiquement dÃ©pondÃ©rÃ©s.
2. **TruncatedSVD** compresse cette matrice creuse en quelques dizaines de dimensions denses, qui captent les "thÃ¨mes" principaux (luxe, proximitÃ©, Ã©quipements, etc.).

> **Anti-leak** : le vectoriseur et le SVD sont entraÃ®nÃ©s UNIQUEMENT sur le train. Le test passe par `transform()`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tfidf_svd_features(train_text, test_text, n_components, prefix, max_features=5000, ngram_range=(1, 2), min_df=10):
    # Fit TF-IDF + SVD UNIQUEMENT sur le train, applique sur le test
    tfidf = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                            min_df=min_df, stop_words='english', lowercase=True)
    X_tr_tfidf = tfidf.fit_transform(train_text.fillna('').astype(str))
    X_te_tfidf = tfidf.transform(test_text.fillna('').astype(str))

    n_components = min(n_components, X_tr_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    X_tr_svd = svd.fit_transform(X_tr_tfidf)
    X_te_svd = svd.transform(X_te_tfidf)

    cols = [f'{prefix}_{i}' for i in range(n_components)]
    train_svd = pd.DataFrame(X_tr_svd, columns=cols, index=train_text.index)
    test_svd  = pd.DataFrame(X_te_svd, columns=cols, index=test_text.index)
    return train_svd, test_svd, cols

# Description : 20 composantes (texte plus riche)
desc_tr, desc_te, DESC_SVD_COLS = tfidf_svd_features(
    train_fe['description'], test_fe['description'],
    n_components=20, prefix='desc_svd', max_features=5000, ngram_range=(1, 2), min_df=10)

# Nom : 5 composantes (texte court)
name_tr, name_te, NAME_SVD_COLS = tfidf_svd_features(
    train_fe['name'], test_fe['name'],
    n_components=5, prefix='name_svd', max_features=2000, ngram_range=(1, 1), min_df=5)

train_fe = pd.concat([train_fe, desc_tr, name_tr], axis=1)
test_fe  = pd.concat([test_fe,  desc_te, name_te], axis=1)

print(f"Description SVD : {len(DESC_SVD_COLS)} composantes")
print(f"Nom SVD         : {len(NAME_SVD_COLS)} composantes")
print(f"train_fe : {train_fe.shape}")
print(f"test_fe  : {test_fe.shape}")

## 4. Encodage des variables catÃ©gorielles

Le modÃ¨le ne peut pas lire des mots comme `New York` ou `Entire home`. On doit les convertir en nombres. Trois approches selon la cardinalitÃ© :

**Quartiers et codes postaux** (centaines de valeurs) â€” deux encodages complÃ©mentaires :
1. **Target encoding** : on remplace chaque quartier par le **prix moyen** des logements de ce quartier. Manhattan devient par exemple 4.8 (le log_price moyen). Pour Ã©viter le data leakage, la moyenne est calculÃ©e en excluant Ã  chaque fois les lignes qu'on est en train de valider (K-fold).
2. **Frequency encoding** : on remplace chaque quartier par son **nombre d'occurrences** dans le train. Ce signal capture la "popularitÃ©" du quartier indÃ©pendamment du prix â€” un quartier avec 10 annonces est probablement trÃ¨s diffÃ©rent d'un quartier avec 1000 annonces.

**Autres catÃ©gories** (type de chambre, ville, type de propriÃ©tÃ©â€¦) â€” **label encoding** :
Un numÃ©ro par valeur : `Entire home` devient 0, `Private room` devient 1, etc. Suffisant car l'arbre LightGBM gÃ¨re bien les variables catÃ©gorielles label-encodÃ©es sur peu de modalitÃ©s.

In [ ]:
def target_encode(train_df, test_df, cols, target='log_price', n_splits=5, smoothing=10):
    global_mean = train_df[target].mean()
    for col in cols:
        enc_tr = np.full(len(train_df), global_mean, dtype=float)
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for tr_idx, val_idx in kf.split(train_df):
            fold_tr  = train_df.iloc[tr_idx]
            fold_val = train_df.iloc[val_idx]
            stats  = fold_tr.groupby(col)[target].agg(['mean','count'])
            smooth = (stats['count']*stats['mean'] + smoothing*global_mean) / (stats['count']+smoothing)
            enc_tr[val_idx] = fold_val[col].map(smooth).fillna(global_mean).values
        stats_all  = train_df.groupby(col)[target].agg(['mean','count'])
        smooth_all = (stats_all['count']*stats_all['mean'] + smoothing*global_mean) / (stats_all['count']+smoothing)
        train_df[col+'_te'] = enc_tr
        test_df[col+'_te']  = test_df[col].map(smooth_all).fillna(global_mean).values
    return train_df, test_df

def frequency_encode(train_df, test_df, cols):
    # Encodage par frÃ©quence : remplace la catÃ©gorie par son nombre d'occurrences
    # dans le train. Capture la "popularitÃ©" du quartier/zip indÃ©pendamment du prix.
    for col in cols:
        freq = train_df[col].value_counts()
        train_df[col+'_fe'] = train_df[col].map(freq).fillna(0).astype(int)
        test_df[col+'_fe']  = test_df[col].map(freq).fillna(0).astype(int)
    return train_df, test_df

def label_encode(train_df, test_df, cols):
    for col in cols:
        le = LabelEncoder()
        combined = pd.concat([train_df[col].fillna('Unknown').astype(str),
                               test_df[col].fillna('Unknown').astype(str)], ignore_index=True)
        le.fit(combined)
        train_df[col+'_le'] = le.transform(train_df[col].fillna('Unknown').astype(str))
        test_df[col+'_le']  = le.transform(test_df[col].fillna('Unknown').astype(str))
    return train_df, test_df

TE_COLS = ['neighbourhood_clean', 'zipcode_clean']
FE_COLS = ['neighbourhood_clean', 'zipcode_clean']
LE_COLS = ['property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city']

train_enc, test_enc = target_encode(train_fe, test_fe, TE_COLS)
train_enc, test_enc = frequency_encode(train_enc, test_enc, FE_COLS)
train_enc, test_enc = label_encode(train_enc, test_enc, LE_COLS)
print("Encodage terminÃ©")

In [ ]:
NUM_FEAT = [
    'accommodates','bathrooms','bedrooms','beds',
    'number_of_reviews','review_scores_rating','latitude','longitude',
    'amenities_count','description_len','description_words','name_len',
    'host_response_rate',
    'host_since_days','first_review_days','last_review_days',
    'host_since_missing','first_review_missing','last_review_missing',
    'review_missing',
    'host_has_profile_pic','host_identity_verified','instant_bookable','cleaning_fee',
    # Interactions de taille (densitÃ© d'occupation)
    'beds_per_person','bath_per_bedroom','bedroom_per_person','has_no_bathroom',
    # GÃ©ographie : distance au centre mÃ©dian de la ville
    'dist_to_center',
]
TE_FEAT  = [c+'_te' for c in TE_COLS]
FE_FEAT  = [c+'_fe' for c in FE_COLS]
LE_FEAT  = [c+'_le' for c in LE_COLS]
AM_FEAT  = [c for c in train_enc.columns if c.startswith('am_')]
TXT_FEAT = DESC_SVD_COLS + NAME_SVD_COLS  # composantes TF-IDF/SVD
ALL_FEAT = NUM_FEAT + TE_FEAT + FE_FEAT + LE_FEAT + AM_FEAT + TXT_FEAT

print(f"Total features : {len(ALL_FEAT)}")
print(f"  NumÃ©riques        : {len(NUM_FEAT)}")
print(f"  Target encoded    : {len(TE_FEAT)}")
print(f"  Frequency encoded : {len(FE_FEAT)}")
print(f"  Label encoded     : {len(LE_FEAT)}")
print(f"  Amenities         : {len(AM_FEAT)}")
print(f"  Texte (SVD)       : {len(TXT_FEAT)}")

In [ ]:
X      = train_enc[ALL_FEAT].copy()
y      = train_enc['log_price'].copy()
X_test = test_enc[ALL_FEAT].copy()

medians = X.median()
X       = X.fillna(medians)
X_test  = X_test.fillna(medians)

print(f"X      : {X.shape}")
print(f"y      : {y.shape}  mean={y.mean():.3f}  std={y.std():.3f}")
print(f"X_test : {X_test.shape}")
print(f"NaN dans X      : {X.isna().sum().sum()}")
print(f"NaN dans X_test : {X_test.isna().sum().sum()}")

## 5. Exploration des ModÃ¨les et Configurations

Avant de choisir le modÃ¨le final, on teste plusieurs approches pour justifier nos choix.

1. **Comparaison de modÃ¨les** : Ridge, Random Forest, Gradient Boosting, LightGBM â€” mÃªme jeu de features, pour voir quel algorithme marche le mieux
2. **Comparaison de configurations** : du jeu de features minimal au complet â€” pour voir l'apport de chaque groupe de variables

On utilise une validation croisÃ©e 3-fold (plus rapide) pour comparer.

In [ ]:
# â”€â”€ Comparaison de modÃ¨les (3-fold CV) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

kf3 = KFold(n_splits=3, shuffle=True, random_state=SEED)

# Liste des modÃ¨les Ã  tester
candidates = {
    'Ridge (linÃ©aire)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10))
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED
    ),
}
if USE_LGBM:
    import lightgbm as lgb
    candidates['LightGBM'] = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        random_state=SEED, verbose=-1, n_jobs=-1
    )

results_models = {}
print('Comparaison des modÃ¨les...')
for name, model in candidates.items():
    scores = []
    for tr_idx, val_idx in kf3.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))
    results_models[name] = np.mean(scores)
    print(f'  {name:<25} RMSE = {np.mean(scores):.4f}')

best_model = min(results_models, key=results_models.get)
print(f'\nMeilleur modÃ¨le : {best_model} (RMSE={results_models[best_model]:.4f})')

In [ ]:
# â”€â”€ Graphique comparaison des modÃ¨les â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(results_models.keys())
scores = list(results_models.values())
best_score = min(scores)
colors = ['#e74c3c' if s == best_score else '#aec6e8' for s in scores]

bars = ax.bar(names, scores, color=colors, edgecolor='white', linewidth=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('RMSE sur log_price â€” plus bas = meilleur')
ax.set_title('Comparaison des modÃ¨les (CV 3-fold, toutes les features)')
ax.set_ylim(min(scores) * 0.97, max(scores) * 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Observations : comparaison des modÃ¨les

- Le modÃ¨le linÃ©aire (Ridge) est clairement moins performant : la relation entre les features et le prix n'est pas linÃ©aire
- Random Forest et Gradient Boosting donnent de bons rÃ©sultats car ils capturent des interactions non-linÃ©aires entre les variables
- LightGBM obtient le meilleur score avec les mÃªmes features â€” c'est donc lui qu'on utilisera pour le modÃ¨le final

On garde LightGBM pour la suite.

In [ ]:
# â”€â”€ Comparaison de configurations de features â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# On part du plus simple et on ajoute des groupes de variables progressivement

FEAT_BASE  = ['accommodates','bathrooms','bedrooms','beds',
              'number_of_reviews','review_scores_rating','latitude','longitude']
FEAT_CAT   = FEAT_BASE + LE_FEAT + TE_FEAT
FEAT_AM    = FEAT_CAT + AM_FEAT
FEAT_FULL  = ALL_FEAT  # tout

configs = {
    'Taille + localisation (8 features)': FEAT_BASE,
    '+ CatÃ©gories encodÃ©es':               FEAT_CAT,
    '+ Amenities (Ã©quipements)':            FEAT_AM,
    'Toutes les features':                  FEAT_FULL,
}

if USE_LGBM:
    base_model = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        random_state=SEED, verbose=-1, n_jobs=-1
    )
else:
    base_model = GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED
    )

results_configs = {}
print('Comparaison des configurations...')
for name, feats in configs.items():
    X_cfg = X[feats].fillna(X[feats].median())
    scores = []
    for tr_idx, val_idx in kf3.split(X_cfg):
        X_tr, X_val = X_cfg.iloc[tr_idx], X_cfg.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        import copy
        m = copy.deepcopy(base_model)
        m.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, m.predict(X_val))))
    results_configs[name] = (np.mean(scores), len(feats))
    print(f'  {name:<40} {len(feats):3d} features  RMSE={np.mean(scores):.4f}')

In [ ]:
# â”€â”€ Graphique : impact des groupes de features â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cfg_names  = list(results_configs.keys())
cfg_rmse   = [v[0] for v in results_configs.values()]
cfg_nfeat  = [v[1] for v in results_configs.values()]
best_rmse  = min(cfg_rmse)
colors = ['#e74c3c' if s == best_rmse else '#aec6e8' for s in cfg_rmse]

# Barres RMSE
bars = axes[0].bar(range(len(cfg_names)), cfg_rmse, color=colors,
                   edgecolor='white', linewidth=0.5)
for bar, score in zip(bars, cfg_rmse):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(range(len(cfg_names)))
axes[0].set_xticklabels(cfg_names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE â€” plus bas = meilleur')
axes[0].set_title('RMSE selon la configuration de features')
axes[0].set_ylim(min(cfg_rmse)*0.97, max(cfg_rmse)*1.02)
axes[0].grid(axis='y', alpha=0.3)

# AmÃ©lioration progressive
axes[1].plot(range(len(cfg_names)), cfg_rmse, 'o-', color='steelblue',
             linewidth=2, markersize=8)
axes[1].fill_between(range(len(cfg_names)), cfg_rmse, alpha=0.15, color='steelblue')
axes[1].set_xticks(range(len(cfg_names)))
axes[1].set_xticklabels(cfg_names, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('RMSE')
axes[1].set_title('AmÃ©lioration progressive en ajoutant des features')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ Tableau rÃ©capitulatif de tous les rÃ©sultats â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rows = []
for name, rmse in results_models.items():
    rows.append({'ExpÃ©rience': name, 'Type': 'ModÃ¨le', 'RMSE (CV 3-fold)': round(rmse, 4)})
for name, (rmse, nf) in results_configs.items():
    rows.append({'ExpÃ©rience': f'{name} ({nf} feat.)', 'Type': 'Config features',
                 'RMSE (CV 3-fold)': round(rmse, 4)})

summary = pd.DataFrame(rows).sort_values('RMSE (CV 3-fold)')
print('=== RÃ©capitulatif des expÃ©riences ===')
print(summary.to_string(index=False))

### Conclusions des expÃ©riences

- **LightGBM** donne le meilleur score parmi tous les modÃ¨les testÃ©s
- Chaque groupe de features supplÃ©mentaire amÃ©liore le score : les catÃ©gories encodÃ©es (quartier, ville) apportent le plus gros gain, suivi des amenities et des features texte/hÃ´te
- Utiliser **toutes les features + LightGBM** est donc la meilleure combinaison

On passe maintenant Ã  l'entraÃ®nement final avec cette configuration.

## 5.5 Tuning des hyperparamÃ¨tres (Optuna)

Les paramÃ¨tres LightGBM utilisÃ©s en section 5 ont Ã©tÃ© choisis Ã  la main. On peut faire mieux en laissant **Optuna** chercher la meilleure combinaison automatiquement.

Optuna fait 50 essais (â‰ˆ 20 min sur CPU). Ã€ chaque essai il propose un jeu de paramÃ¨tres et mesure le RMSE par validation croisÃ©e 3-fold (3 folds pour aller plus vite). L'algorithme **TPE** apprend des essais prÃ©cÃ©dents pour viser les zones prometteuses au lieu de tirer au hasard.

Espace explorÃ© :
- `learning_rate` âˆˆ [0.01, 0.1]
- `num_leaves` âˆˆ [31, 255]
- `max_depth` âˆˆ {-1, 4, 6, 8, 10}
- `min_child_samples` âˆˆ [5, 100]
- `feature_fraction`, `bagging_fraction` âˆˆ [0.5, 1.0]
- `reg_alpha`, `reg_lambda` âˆˆ [1e-3, 10] (log)

Les meilleurs paramÃ¨tres trouvÃ©s sont ensuite repris en section 6 pour le modÃ¨le final.

In [ ]:
# â”€â”€ Tuning des hyperparamÃ¨tres LightGBM avec Optuna â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Optuna explore l'espace des hyperparamÃ¨tres avec un Ã©chantillonneur bayÃ©sien
# (TPE), bien plus efficace qu'une grille alÃ©atoire. Si Optuna n'est pas
# installÃ©, on garde les paramÃ¨tres manuels utilisÃ©s en section 5.

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
    print(f"Optuna {optuna.__version__} disponible â€” lancement du tuning")
except ImportError:
    HAS_OPTUNA = False
    print("Optuna non installÃ© â€” `pip install optuna` pour activer le tuning")
    print("On garde les paramÃ¨tres manuels (learning_rate=0.05, num_leaves=63, ...)")

best_params = None

if HAS_OPTUNA and USE_LGBM:
    kf3_tune = KFold(n_splits=3, shuffle=True, random_state=SEED)

    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'verbose': -1,
            'random_state': SEED,
            'n_jobs': -1,
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
            'max_depth':         trial.suggest_categorical('max_depth', [-1, 4, 6, 8, 10]),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq':      5,
            'reg_alpha':         trial.suggest_float('reg_alpha',  1e-3, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        scores = []
        for tr_idx, val_idx in kf3_tune.split(X):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            dtr = lgb.Dataset(X_tr, label=y_tr)
            dvl = lgb.Dataset(X_val, label=y_val)
            m = lgb.train(params, dtr, num_boost_round=1500,
                          valid_sets=[dvl],
                          callbacks=[lgb.early_stopping(50, verbose=False)])
            scores.append(np.sqrt(mean_squared_error(y_val, m.predict(X_val))))
        return float(np.mean(scores))

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    best_params = study.best_params
    print(f"\nMeilleur RMSE (3-fold) : {study.best_value:.4f}")
    print(f"Meilleurs paramÃ¨tres :")
    for k, v in best_params.items():
        print(f"  {k:<20} = {v}")

## 6. ModÃ¨le Final

### Pourquoi LightGBM ?
C'est un algorithme qui construit plein de petits arbres de dÃ©cision Ã  la suite, chacun corrigeant les erreurs du prÃ©cÃ©dent. On l'a choisi car il donne de trÃ¨s bons rÃ©sultats sur des donnÃ©es tabulaires comme les nÃ´tres, et il gÃ¨re bien les valeurs manquantes.

### HyperparamÃ¨tres
On utilise les **paramÃ¨tres trouvÃ©s par Optuna** en section 5.5. Si Optuna n'a pas pu tourner (paquet absent), on retombe sur des valeurs manuelles raisonnables (`learning_rate=0.05`, `num_leaves=63`, etc.).

### Validation croisÃ©e (5-fold)
Pour Ã©valuer le modÃ¨le sans toucher aux donnÃ©es de test, on divise le train en 5 parties Ã©gales. On entraÃ®ne 5 fois : Ã  chaque fois, 4 parties servent Ã  apprendre et la 5Ã¨me sert Ã  mesurer la performance. Cela donne une note fiable sur des donnÃ©es que le modÃ¨le n'a pas vues.

L'entraÃ®nement s'arrÃªte automatiquement quand les performances ne s'amÃ©liorent plus â€” pour Ã©viter de surapprendre.

### Multi-seed averaging
Le modÃ¨le final est en rÃ©alitÃ© **3 modÃ¨les entraÃ®nÃ©s avec 3 seeds diffÃ©rentes**, dont les prÃ©dictions sont moyennÃ©es. Comme LightGBM Ã©chantillonne alÃ©atoirement les features et les lignes Ã  chaque arbre, changer la seed donne des modÃ¨les lÃ©gÃ¨rement diffÃ©rents. La moyenne rÃ©duit la variance des prÃ©dictions sans changer le biais.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
cv_scores = []

if USE_LGBM:
    # Si Optuna a tournÃ© en section 5.5, on prend ses meilleurs paramÃ¨tres.
    # Sinon, on retombe sur les paramÃ¨tres manuels initiaux.
    if best_params is not None:
        params = {
            'objective': 'regression', 'metric': 'rmse',
            'verbose': -1, 'random_state': SEED, 'n_jobs': -1,
            'bagging_freq': 5,
            **best_params,
        }
        print(f"ParamÃ¨tres LightGBM : Optuna ({len(best_params)} hyperparams optimisÃ©s)")
    else:
        params = dict(
            objective='regression', metric='rmse',
            learning_rate=0.05, num_leaves=63, max_depth=-1,
            min_child_samples=20, feature_fraction=0.8,
            bagging_fraction=0.8, bagging_freq=5,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=SEED, verbose=-1, n_jobs=-1,
        )
        print("ParamÃ¨tres LightGBM : manuels (Optuna non utilisÃ©)")

    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=ALL_FEAT)
        dval   = lgb.Dataset(X_val, label=y_val)
        model  = lgb.train(params, dtrain, num_boost_round=3000,
                           valid_sets=[dval],
                           callbacks=[lgb.early_stopping(150, verbose=False),
                                      lgb.log_evaluation(500)])
        preds = model.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse); best_iters.append(model.best_iteration)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f} | best_iter={model.best_iteration}")

    best_n = int(np.mean(best_iters) * 1.05)
    print(f"\nMoyenne best_iter={int(np.mean(best_iters))} â†’ modÃ¨le final : {best_n} rounds")

else:
    from sklearn.ensemble import GradientBoostingRegressor
    best_n = 300
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                       max_depth=5, subsample=0.8, random_state=SEED)
        m.fit(X_tr, y_tr); preds = m.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f}")

# RÂ² calculÃ© Ã  partir du RMSE moyen et de la variance de y
mean_rmse = float(np.mean(cv_scores))
r2_cv = 1 - (mean_rmse ** 2) / float(y.var())

print(f"\n{'='*40}")
print(f"CV RMSE : {mean_rmse:.4f} Â± {np.std(cv_scores):.4f}")
print(f"CV RÂ²   : {r2_cv:.4f}")
print(f"{'='*40}")

### RÃ©sultats de la validation croisÃ©e

Un RMSE d'environ 0.44 sur le log_price correspond Ã  une erreur d'environ 55% sur le prix rÃ©el â€” ce qui est attendu pour ce type de donnÃ©es trÃ¨s variables.

Les scores sont similaires sur les 5 folds, donc le modÃ¨le est stable. Le modÃ¨le final est ensuite entraÃ®nÃ© sur toutes les donnÃ©es disponibles.

In [ ]:
# ModÃ¨le final entraÃ®nÃ© sur tout le train, moyennÃ© sur 3 seeds.
# Multi-seed averaging : chaque modÃ¨le voit un sous-Ã©chantillonnage lÃ©gÃ¨rement
# diffÃ©rent (bagging_fraction=0.8 + bagging_freq=5), donc moyenner rÃ©duit la
# variance des prÃ©dictions sans coÃ»t de tuning supplÃ©mentaire.

SEEDS = [42, 123, 2024]
print(f"EntraÃ®nement final ({best_n} itÃ©rations Ã— {len(SEEDS)} seeds)...")

if USE_LGBM:
    test_preds = np.zeros(len(X_test))
    for s in SEEDS:
        params_s = {**params, 'random_state': s}
        dtrain_full = lgb.Dataset(X, label=y, feature_name=ALL_FEAT)
        m = lgb.train(params_s, dtrain_full, num_boost_round=best_n)
        test_preds += m.predict(X_test) / len(SEEDS)
        print(f"  seed={s} OK")
    final_model = m  # le dernier, pour feature_importance plus bas
else:
    final_model = GradientBoostingRegressor(n_estimators=best_n, learning_rate=0.1,
                                             max_depth=5, subsample=0.8, random_state=SEED)
    final_model.fit(X, y)
    test_preds = final_model.predict(X_test)

print(f"\nPrÃ©dictions test â€” min={test_preds.min():.3f}  max={test_preds.max():.3f}  "
      f"mean={test_preds.mean():.3f}")

## 7. Analyse du ModÃ¨le

On vÃ©rifie que le modÃ¨le a bien appris en regardant :
1. **Quelles variables ont Ã©tÃ© les plus utiles ?**
2. **Les prÃ©dictions sont-elles proches des vraies valeurs ?**
3. **Les erreurs sont-elles alÃ©atoires** ou y a-t-il un pattern ?

In [ ]:
# Feature importance
if USE_LGBM:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importance(importance_type='gain')})
else:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importances_})

imp = imp.sort_values('importance', ascending=False).reset_index(drop=True)
top20 = imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top20)))
ax.barh(range(len(top20)), top20['importance'].values, color=colors)
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['feature'].values, fontsize=10)
ax.invert_yaxis(); ax.set_title('Top 20 features (importance gain)', fontsize=13)
plt.tight_layout(); plt.show()

print("Top 10 features:")
print(imp.head(10).to_string(index=False))

### Ce que nous dit l'importance des features

- **La localisation** (quartier, code postal, latitude/longitude) est de loin la variable la plus importante â€” cohÃ©rent avec l'EDA
- **La taille** du logement (capacitÃ©, chambres, salles de bain) arrive en deuxiÃ¨me position
- **L'anciennetÃ©** de l'annonce (date du premier avis, date d'inscription) a aussi un impact
- Certains **Ã©quipements** apparaissent dans le top 20, ce qui confirme que les extraire Ã©tait pertinent

In [ ]:
# OOF : prÃ©dictions vs rÃ©el + distribution des rÃ©sidus
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y, oof_preds, alpha=0.1, s=1, color='steelblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Parfait')
axes[0].set_xlabel('log_price rÃ©el'); axes[0].set_ylabel('log_price prÃ©dit (OOF)')
axes[0].set_title('OOF : prÃ©dit vs rÃ©el'); axes[0].legend()

residuals = y - oof_preds
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white', lw=0.5)
axes[1].axvline(0, color='black', ls='--')
axes[1].set_title(f'RÃ©sidus (std={residuals.std():.3f})')
axes[1].set_xlabel('RÃ©sidu (rÃ©el âˆ’ prÃ©dit)')

plt.tight_layout(); plt.show()
oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"OOF RMSE global : {oof_rmse:.4f}")

### InterprÃ©tation des rÃ©sidus

- Les points s'alignent bien sur la diagonale : les prÃ©dictions sont globalement proches des vraies valeurs
- Les erreurs sont rÃ©parties de faÃ§on symÃ©trique autour de 0 : le modÃ¨le ne surestime pas systÃ©matiquement
- Le modÃ¨le est un peu moins prÃ©cis sur les prix trÃ¨s Ã©levÃ©s ou trÃ¨s bas, ces cas Ã©tant moins reprÃ©sentÃ©s dans les donnÃ©es

## 8. Export des prÃ©dictions

In [ ]:
submission = pd.DataFrame({'logpred': test_preds}, index=test_enc['id'].values)
submission.index.name = ''
submission.to_csv('MaPredictionFinale.csv')
print(f"SauvegardÃ© : MaPredictionFinale.csv  ({len(submission):,} lignes)")
submission.head()

In [ ]:
# VÃ©rification du format avec la mÃªme logique que estConforme() de example.ipynb
def estConforme(monFichier_csv):
    votre_prediction = pd.read_csv(monFichier_csv)
    fichier_exemple  = pd.read_csv("prediction_example.csv")
    assert votre_prediction.columns[1] == fichier_exemple.columns[1], \
        f"Colonne prÃ©diction doit s'appeler {fichier_exemple.columns[1]}"
    assert len(votre_prediction) == len(fichier_exemple), \
        f"Doit avoir {len(fichier_exemple)} prÃ©dictions"
    assert np.all(votre_prediction.iloc[:, 0] == fichier_exemple.iloc[:, 0]), \
        "Les ids doivent correspondre Ã  ceux de prediction_example.csv"
    print("Format conforme : OK")

estConforme('MaPredictionFinale.csv')

example = pd.read_csv('prediction_example.csv', index_col=0)
pred    = pd.read_csv('MaPredictionFinale.csv', index_col=0)

print(f"\n=== VÃ©rifications complÃ©mentaires ===")
print(f"Colonnes attendues : {list(example.columns)}")
print(f"Colonnes obtenues  : {list(pred.columns)}")
print(f"Lignes             : {len(pred):,}")
print(f"NaN dans logpred   : {pred['logpred'].isna().sum()}")
print(f"Range logpred      : [{pred['logpred'].min():.3f}, {pred['logpred'].max():.3f}]")
print(f"Moyenne logpred    : {pred['logpred'].mean():.3f}  (train: {y.mean():.3f})")

assert pred['logpred'].isna().sum() == 0, "Il y a des NaN dans logpred"
assert pred['logpred'].between(2, 9).all(), "Des prÃ©dictions hors range plausible"

print("\n=== AperÃ§u ===")
print(pred.head(5))

plt.figure(figsize=(10, 4))
plt.hist(pred['logpred'], bins=60, color='steelblue', edgecolor='white', lw=0.5, label='Test (prÃ©dit)')
plt.axvline(y.mean(), color='red', ls='--', label=f'Moyenne train ({y.mean():.2f})')
plt.title('Distribution des prÃ©dictions'); plt.xlabel('log_price prÃ©dit')
plt.legend(); plt.tight_layout(); plt.show()
print("\nExport OK !")

## 9. Conclusion

### Pipeline
```
EDA -> Feature engineering (interactions, gÃ©o, dates) -> TF-IDF/SVD texte
    -> Encodage (target + frequency + label) -> Tuning Optuna
    -> LightGBM 5-fold + multi-seed averaging -> Export
```

### RÃ©sultats
- RMSE et RÂ² stables sur les 5 folds (voir section 6 pour les valeurs exactes)
- Variables les plus utiles : la localisation (target encoding du quartier, coordonnÃ©es GPS) et la taille du logement

### Ce qui a marchÃ©
- **Target encoding** des quartiers et codes postaux â€” gros gain par rapport au label encoding seul
- **Top 50 amenities** + frequency encoding apportent des signaux complÃ©mentaires
- **TF-IDF + SVD** sur la description capte des thÃ¨mes (luxe, proximitÃ©â€¦) que les longueurs de texte ne voient pas
- **Optuna** explore mieux l'espace des hyperparamÃ¨tres qu'un tuning manuel
- **Multi-seed averaging** rÃ©duit la variance des prÃ©dictions Ã  coÃ»t quasi nul

### Pistes d'amÃ©lioration restantes
- Embeddings sentence-transformers pour la description (gain marginal mais possible)
- Stacking de plusieurs familles de modÃ¨les (LightGBM + CatBoost + XGBoost)
- DonnÃ©es externes sur les quartiers (transports, restaurants, Ã©colesâ€¦)